In [1]:
from PIL import Image
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from os.path import join, exists
import pickle
import time
from copy import deepcopy
import random

import torch
from src.open_clip.factory import create_model_and_transforms, get_tokenizer
from torch.utils.data import Dataset, DataLoader
from src.training.precision import get_autocast
from pathlib import Path


In [2]:
# Mode config:
# - 'benchmark': use benchmark CSV and compute retrieval metrics (R@K)
# - 'image_dir': no CSV required, build image index directly from a folder
run_mode = 'image_dir'

# Root data directory used when CSV filepath is relative (benchmark mode)
ROOT_DATA_DIR = '/PATH/TO/BENCHMARK_DATASETS_ROOT'

# Image-directory mode settings (your case)
IMAGE_ROOT_DIR = '/scratch/groups/smfletch/4new_Mofan_RS'
IMAGE_EXTENSIONS = ('.jpg', '.jpeg')
RECURSIVE_SCAN = True  # True=递归扫描（包含所有子文件夹）; False=只扫描当前目录


In [3]:
# Benchmark CSV paths (only used when run_mode='benchmark')
data_csv_path_dict = {
    'SkyScript': '/PATH/TO/CSV/SkyScript_test_30K_filtered_by_CLIP_openai.csv',
    'RSICD': '/PATH/TO/CSV/RSICD/RSICD_img_txt_pairs_test.csv',
    'RSITMD': '/PATH/TO/CSV/RSITMD/RSITMD_img_txt_pairs_test.csv',
    'ucmcaptions': '/PATH/TO/CSV/ucmcaptions/ucmcaptions_img_txt_pairs_test.csv',
}


In [4]:
batch_size = 128
precision = 'amp'
autocast = get_autocast(precision)

In [5]:
class CsvDataset_customized(Dataset):
    def __init__(self, df, transforms, img_key, caption_key, tokenizer=None, return_img_path=False, 
                 root_data_dir=None):
        if root_data_dir is not None:
            df[img_key] = df[img_key].apply(lambda x: join(root_data_dir, x))
        self.images = df[img_key].tolist()
        self.captions = df[caption_key].tolist()
        self.transforms = transforms
        self.tokenize = tokenizer
        self.return_img_path = return_img_path

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):
        images = self.transforms(Image.open(str(self.images[idx])))
        texts = self.tokenize([str(self.captions[idx])])[0]
        if self.return_img_path:
            return images, texts, str(self.images[idx])
        return images, texts
    
class CsvDataset_image(Dataset):
    def __init__(self, df, transforms, img_key, return_img_path=False, root_data_dir=None):
        if root_data_dir is not None:
            df[img_key] = df[img_key].apply(lambda x: join(root_data_dir, x))
        self.images = df[img_key].tolist()
        self.transforms = transforms
        self.return_img_path = return_img_path

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        images = self.transforms(Image.open(str(self.images[idx])))
        if self.return_img_path:
            return images, str(self.images[idx])
        return images
    
class CsvDataset_text(Dataset):
    def __init__(self, df, caption_key, tokenizer=None, return_original_text=False, root_data_dir=None):
        if root_data_dir is not None:
            df[img_key] = df[img_key].apply(lambda x: join(root_data_dir, x))
        self.captions = df[caption_key].tolist()
        self.tokenize = tokenizer
        self.return_original_text = return_original_text

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):
        original_text = str(self.captions[idx])
        texts = self.tokenize([original_text])[0]
        if self.return_original_text:
            return texts, original_text
        return texts

def random_seed(seed=42, rank=0):
    torch.manual_seed(seed + rank)
    np.random.seed(seed + rank)
    random.seed(seed + rank)

def get_sample_identifier(filepath):
    return '/'.join(filepath.split('/')[-2:])


In [32]:
model_arch_name = 'ViT-L-14' # or 'ViT-B-32'
# Download checkpoints from README and unzip; then point to the .pt/.bin checkpoint file
ckpt_name = '/PATH/TO/MODEL_CHECKPOINT.pt'

# only used in benchmark mode
dataset_name = 'ucmcaptions'  # choose from keys in data_csv_path_dict
data_csv_path = data_csv_path_dict.get(dataset_name, None)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
print(f'run_mode={run_mode}')


cuda


In [33]:
if dataset_name == 'SkyScript':
    caption_key = 'title_multi_objects'
else:
    caption_key = 'title'

force_quick_gelu = True

random_seed(42, 0)
if 'ViT-B-32' in model_arch_name:
    model, _, preprocess_val = create_model_and_transforms(
            model_arch_name,
            ckpt_name,
            precision=precision,
            device=device,
            output_dict=True,
        )
else:
    model, _, preprocess_val = create_model_and_transforms(
            model_arch_name,
            ckpt_name,
            precision=precision,
            device=device,
            output_dict=True,
            force_quick_gelu=force_quick_gelu,
        )

tokenizer = get_tokenizer(model_arch_name)


In [34]:
if run_mode == 'benchmark':
    df = pd.read_csv(data_csv_path)
    # If filepath is relative, join with ROOT_DATA_DIR. Keep absolute paths unchanged.
    df['filepath'] = df['filepath'].apply(lambda x: x if os.path.isabs(str(x)) else join(ROOT_DATA_DIR, str(x)))
    if dataset_name in ['RSICD', 'RSITMD', 'ucmcaptions']:
        df[caption_key] = df[caption_key].apply(lambda x: 'a satellite image. ' + x)
else:
    pattern = '**/*' if RECURSIVE_SCAN else '*'
    image_paths = []
    for fp in Path(IMAGE_ROOT_DIR).glob(pattern):
        if fp.is_file() and fp.suffix.lower() in IMAGE_EXTENSIONS:
            image_paths.append(str(fp))
    image_paths = sorted(image_paths)
    assert len(image_paths) > 0, f'No images found in {IMAGE_ROOT_DIR} with ext={IMAGE_EXTENSIONS}'
    df = pd.DataFrame({'filepath': image_paths})

print('num_rows:', len(df))
print(df.head(3))


In [37]:
df_image = df[['filepath']].drop_duplicates().reset_index(drop=True)

if run_mode == 'benchmark':
    df_text = df[[caption_key]].drop_duplicates().reset_index(drop=True)
    print('unique images:', df_image.shape, '| unique texts:', df_text.shape)
else:
    df_text = None
    print('unique images:', df_image.shape)


((210, 3), (377, 3))

In [38]:
# unique images
dataset_image = CsvDataset_image(
    df=df_image, 
    transforms=preprocess_val,
    img_key='filepath',
    return_img_path=True,
    
)

dataloader = DataLoader(dataset_image, batch_size=batch_size, shuffle=False, num_workers=4)

model.eval()
all_image_features = []
all_image_paths = []
with torch.no_grad():
    for batch in tqdm(dataloader, unit_scale=batch_size):
        images, img_paths = batch
        images = images.to(device=device)
        with autocast():
            image_features = model.encode_image(images, normalize=True)
            all_image_features.append(image_features.cpu())
            all_image_paths.extend(img_paths)
all_image_features = torch.cat(all_image_features)

100%|█████████████████████████████████████████████████████████████████████████████| 256/256 [00:02<00:00, 105.62it/s]


In [39]:
if run_mode == 'benchmark':
    # unique texts
    dataset_text = CsvDataset_text(
        df=df_text,
        caption_key=caption_key,
        tokenizer=tokenizer,
        return_original_text=True,
    )

    dataloader = DataLoader(dataset_text, batch_size=batch_size, shuffle=False, num_workers=4)

    model.eval()
    all_text_features = []
    all_texts = []
    with torch.no_grad():
        for batch in tqdm(dataloader, unit_scale=batch_size):
            texts, original_texts = batch
            texts = texts.to(device=device)
            with autocast():
                text_features = model.encode_text(texts, normalize=True)
                all_text_features.append(text_features.cpu())
                all_texts.extend(original_texts)
    all_text_features = torch.cat(all_text_features)
else:
    all_text_features = None
    all_texts = None
    print('Skip text bank encoding in image_dir mode.')


100%|█████████████████████████████████████████████████████████████████████████████| 384/384 [00:00<00:00, 542.15it/s]


In [40]:
if run_mode == 'benchmark':
    text_indices = {x: i for i, x in enumerate(all_texts)}
    img_indices = {x: i for i, x in enumerate(all_image_paths)}
else:
    text_indices = None
    img_indices = {x: i for i, x in enumerate(all_image_paths)}


In [41]:
if run_mode == 'benchmark':
    # ground truth
    img_path2text = {}
    text2img_path = {}
    for i in tqdm(df.index):
        text = df.loc[i, caption_key]
        img_path = df.loc[i, 'filepath']
        text_id = text_indices[text]
        img_id = img_indices[img_path]
        if img_path not in img_path2text:
            img_path2text[img_path] = set()
        img_path2text[img_path].add(text_id)
        if text not in text2img_path:
            text2img_path[text] = set()
        text2img_path[text].add(img_id)
else:
    img_path2text = None
    text2img_path = None


100%|█████████████████████████████████████████████████████████████████████████| 1050/1050 [00:00<00:00, 44819.60it/s]


In [42]:
if run_mode == 'benchmark':
    res = {'text2img_R@' + str(k): 0 for k in [1, 5, 10, 100]}
    res.update({'img2text_R@' + str(k): 0 for k in [1, 5, 10, 100]})
else:
    res = {'info': 'R@K metrics are skipped in image_dir mode (no ground-truth captions).'}


In [43]:
if run_mode == 'benchmark':
    # text to image
    logit_scale = 100
    for i in tqdm(range(len(all_texts))):
        text_feature = all_text_features[i]
        logits = logit_scale * text_feature @ all_image_features.t()
        ranking = torch.argsort(logits, descending=True).cpu().numpy()
        for k in [1, 5, 10, 100]:
            intersec = set(ranking[:k]) & set(text2img_path[all_texts[i]])
            if intersec:
                res['text2img_R@' + str(k)] += 1
    for k in [1, 5, 10, 100]:
        res['text2img_R@' + str(k)] /= len(all_texts)
else:
    print('Skip benchmark text->image R@K in image_dir mode.')


100%|████████████████████████████████████████████████████████████████████████████| 377/377 [00:00<00:00, 6980.54it/s]


In [44]:
if run_mode == 'benchmark':
    # image to text
    logit_scale = 100
    for i in tqdm(range(len(all_image_paths))):
        image_feature = all_image_features[i]
        logits = logit_scale * image_feature @ all_text_features.t()
        ranking = torch.argsort(logits, descending=True).cpu().numpy()
        for k in [1, 5, 10, 100]:
            intersec = set(ranking[:k]) & img_path2text[all_image_paths[i]]
            if intersec:
                res['img2text_R@' + str(k)] += 1
    for k in [1, 5, 10, 100]:
        res['img2text_R@' + str(k)] /= len(all_image_paths)
else:
    print('Skip benchmark image->text R@K in image_dir mode.')


100%|████████████████████████████████████████████████████████████████████████████| 210/210 [00:00<00:00, 5743.22it/s]


In [45]:
res


{'text2img_R@1': 0.3183023872679045,
 'text2img_R@5': 0.6419098143236074,
 'text2img_R@10': 0.8196286472148541,
 'text2img_R@100': 0.9973474801061007,
 'img2text_R@1': 0.38571428571428573,
 'img2text_R@5': 0.8428571428571429,
 'img2text_R@10': 0.9380952380952381,
 'img2text_R@100': 1.0}

## Step-by-step: how to run zero-shot retrieval and your own text search
1. **Download benchmark datasets** and unzip them under one root directory (example: `/data/rs_benchmarks`).
2. **Download model checkpoint** from README links, unzip it, and locate the actual checkpoint file (for example `.pt`).
3. In this notebook, set:
   - `ROOT_DATA_DIR` to your benchmark root directory.
   - `data_csv_path_dict[...]` to your local CSV paths.
   - `ckpt_name` to your local model checkpoint file path.
4. Choose `dataset_name` (for benchmark testing use: `SkyScript`, `RSICD`, `RSITMD`, `ucmcaptions`).
5. Run all cells up to feature extraction (`all_image_features`, `all_text_features`) and metric computation (`res`) for standard zero-shot cross-modal retrieval.
6. For **your own dataset**:
   - Prepare a CSV with at least columns: `filepath`, `title`.
   - Add your CSV path into `data_csv_path_dict` (e.g., `'my_dataset': '/path/to/my_dataset.csv'`).
   - Set `dataset_name='my_dataset'`.
7. In the custom query cell, set `query_text` to your phrase (example: `solar panel`), run the cell, and read Top-K retrieved image paths/scores.
8. (Optional) Run visualization cell to display Top-9 retrieved images.

> CSV note: `filepath` can be relative paths under `ROOT_DATA_DIR`, or absolute paths (if absolute, skip extra path-join logic).


## Custom text-to-image retrieval on your own images\n- **image_dir mode** (recommended for your case): no CSV required. The notebook scans `IMAGE_ROOT_DIR` (supports recursive scan), encodes all images, then runs free-text retrieval.\n- **benchmark mode**: requires CSV with `filepath` + caption column (`title` or `title_multi_objects`) to compute R@K metrics.\n\nFor image_dir mode, you still get:\n1. Top-K image paths + similarity scores\n2. Top-K thumbnail visualization in notebook\n
**什么是递归扫描（recursive scan）？**
- 递归扫描：会从 `IMAGE_ROOT_DIR` 开始，继续进入每一层子目录，把所有符合后缀（如 `.jpg`）的图片都收集进索引。
- 非递归扫描：只读取 `IMAGE_ROOT_DIR` 这一层，不会进入子目录。
- 你的场景如果图片可能放在子文件夹里，建议保持 `RECURSIVE_SCAN = True`。


### Quick note on testing command location
Run validation commands such as JSON parsing from the **repository root** (`/workspace/rs-search`) in a terminal, for example:
`python - <<'PY'\nimport json\njson.load(open('cross_modal_retrieval.ipynb'))\nprint('notebook json ok')\nPY`


In [ ]:
# Example: text query -> retrieve Top-K images
# Set your own query here, e.g. 'solar panel', 'airport', 'wind farm'
query_text = 'solar'
top_k = 20

assert query_text.strip() != '', 'query_text cannot be empty.'

model.eval()
with torch.no_grad():
    query_tokens = tokenizer([query_text]).to(device)
    query_features = model.encode_text(query_tokens)
    query_features = query_features / query_features.norm(dim=-1, keepdim=True)

# all_image_features should already be computed in the previous cells
scores = (100.0 * query_features @ all_image_features.t()).squeeze(0)
topk_idx = torch.topk(scores, k=min(top_k, len(all_image_paths))).indices.cpu().numpy()

topk_paths = [all_image_paths[i] for i in topk_idx]
topk_scores = [scores[i].item() for i in topk_idx]

# Output 1: path + similarity score
retrieval_df = pd.DataFrame({
    'rank': np.arange(1, len(topk_paths) + 1),
    'filepath': topk_paths,
    'score': topk_scores,
})
print(retrieval_df.to_string(index=False))
retrieval_df.head(20)


In [ ]:
# Output 2: visualize Top-K thumbnails (up to 25)
import math
import matplotlib.pyplot as plt

n_show = min(len(topk_paths), top_k, 25)
cols = 5
rows = math.ceil(n_show / cols)
plt.figure(figsize=(4 * cols, 4 * rows))
for i in range(n_show):
    plt.subplot(rows, cols, i + 1)
    img = Image.open(topk_paths[i]).convert('RGB')
    plt.imshow(img)
    plt.title(f'#{i+1}\n{topk_scores[i]:.3f}', fontsize=10)
    plt.axis('off')
plt.suptitle(f"Query: '{query_text}' | Top-{n_show}")
plt.tight_layout()
plt.show()
